# Investigating the next-token behaviour and the tokenisation (diagnostic)
Pick **model, prompt, chat, permutations and calibration** in section 2, then run the sections below.
Uses **dev data only** (public train); the public test set is never used here. Runtime: T4 GPU. Secrets: `HF_TOKEN`.

| Section | What it shows |
|---|---|
| 3 | load the model and the dev questions |
| 4 | **tokenisation**: how the prompt, a sentence and the answer digits are tokenised; the same text across the three tokenizers |
| 5 | **scoring table**: the real scoring path (your options) on the first N questions |
| 6 | **next-token detail** per question: top tokens, digit candidates, calibration, greedy continuation |
| 7 | summary of the top-1 next tokens |

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Sinhala MMLU"
!pip -q install -r requirements-colab.txt
import os, sys
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
sys.path.insert(0, 'src')

## 2. Choose the options

In [ ]:
MODEL_KEY = 'qwen'            # 'qwen' (Qwen3-0.6B) | 'gemma' (gemma-3-1b-it) | 'llama' (Llama-3.2-1B-Instruct)
PROMPT = 'nosubject_strict'   # 'nosubject' | 'nosubject_strict' | 'subject_si' | 'subject_en'
CHAT = True                   # True = chat template, False = raw prompt (needs a chat-capable model for True)
PERMS = 1                     # 1 = original option order, 4 = average over the 4 cyclic orders
CALIBRATE = False             # True = subtract the content-free prior (mean of 'N/A', '', '[MASK]')

N = 20                        # first N dev questions for the scoring table and the per-question detail
QUESTION_INDEX = 0            # question used in the tokenisation section (0 .. 1850)
PERM_INDEX = 0                # option order to inspect in sections 4 and 6: 0 original, 1-3 rotated
GENERATE = 100                # tokens of greedy continuation in section 6 (0 = off); only produced when PERMS = 1 and CALIBRATE = False

## 3. Load the model and the dev questions

In [ ]:
import debug_next_token as D
from data_utils import load_jsonl
P = D.preset(MODEL_KEY)
if CHAT and not P['chat_capable']:
    raise ValueError(P['model'] + ' is not chat capable: set CHAT = False')
rows = load_jsonl('dev/dev_easy_full.jsonl')
tok, model = D.load(P['model'], P['dtype'])
row = rows[QUESTION_INDEX]
print(P['model'], '| dtype', P['dtype'], '| prompt', PROMPT, '| chat', CHAT, '| perms', PERMS, '| calibrate', CALIBRATE)
print()
print('Question', QUESTION_INDEX, '|', row['subject'], '| gold answer:', row['answer'])
print(row['question'])
for i, c in enumerate(row['choices'], 1): print(i, c)

## 4. Tokenisation
**4a. The exact prompt, token by token.** Tokens per line of the prompt, then the first and last tokens of the full text the model receives (chat template and special tokens included). `partial bytes` / `byte fallback` mean a Sinhala character is split into raw bytes: the tokenizer does not know it as a unit.

In [ ]:
ids = D.show_prompt_tokens(tok, row, prompt_mode=PROMPT, chat=CHAT, perm_index=PERM_INDEX)

**4b. Word by word (fertility).** For the question text: how many tokens each word costs and which pieces they are.

In [ ]:
D.word_view(tok, row['question'], n_words=20)

**4c. The answer tokens the scorer is looking for.** The scorer needs the digit `1`-`4` to be **one token**. If `' 1'` (space + digit) is two tokens, the model may emit the space first and the digit one step later.

In [ ]:
D.answer_tokenization(tok)

**4d. The same question through all three tokenizers** (tokenizers only, no model weights). Compare the tokens per word.

In [ ]:
D.compare_tokenizers(['Qwen/Qwen3-0.6B', 'google/gemma-3-1b-it', 'meta-llama/Llama-3.2-1B-Instruct'], row['question'], n_words=10)

## 5. Scoring table (the real scoring path with your options)
Same digit tokens, same permutation averaging and calibration as the grid, on the first N questions. Columns: gold, prediction, probabilities, **digit_mass** (probability on the digits at the scored position), the model's **top-1 next token**, and with `PERMS = 4` the prediction of each option order (mapped back to the original numbering).

In [ ]:
df = D.scoring_table(tok, model, rows, n=N, prompt_mode=PROMPT, chat=CHAT, perms=PERMS, calibrate=CALIBRATE)
display(df)

## 6. Next-token detail per question (first N)
For each question, in the option order `PERM_INDEX`: exact text, last input tokens, **top 15 next tokens**, digit candidates (`'1'` vs `' 1'`), the **calibration** steps if `CALIBRATE = True`, and the greedy continuation.

In [ ]:
for i, r in enumerate(rows[:N]):
    print('#' * 100)
    print(f'QUESTION {i}  |  {r["subject"]}  |  gold answer (original order): {r["answer"]}')
    D.show_question(tok, model, r, prompt_mode=PROMPT, chat=CHAT, perm_index=PERM_INDEX, calibrate=CALIBRATE, generate=GENERATE if (PERMS == 1 and not CALIBRATE) else 0)

## 7. Summary over the same N questions
Most common top-1 next token and the mean probability on the digit tokens.

In [ ]:
D.summary(tok, model, rows[:N], prompt_mode=PROMPT, chat=CHAT)

## How to read the results
| You see | Meaning | Likely fix |
|---|---|---|
| 4c: `' 1'` is **two tokens** and the top-1 next token is a space | the answer is space + digit; the scorer reads the position **before** the space | score the two-token answer |
| 4a/4b: many `partial bytes` tokens, high tokens per word | the tokenizer splits Sinhala into bytes (Qwen / Llama); long prompts, little Sinhala knowledge | prefer Gemma-family tokenizers |
| Top-1 is a Sinhala word or the question again | the model does not start with the digit | change the instruction / prefill the answer |
| Top-1 is a thinking tag (`<think>`) | thinking is on in the chat template | check `enable_thinking=False` |
| digit_mass high (> 0.9) but accuracy ~ 25% | the model answers in format but does not know the answer | model size / Sinhala knowledge: not a format problem |
| `pred_per_order` differs between orders | the model's answer depends on the position of the option | permutation averaging helps; positional bias |
| calibration changes the prediction a lot | the model's digit prior is strong | compare calibrated and uncalibrated accuracy on dev |